# REC Forecast Generation: Corrected Meter Readings

This notebook generates forecasted "corrected meter readings" for REC members after internal energy sharing of the current forecast.

## Why Compute Corrected Forecasts from Forecast Data?

We apply REC sharing formulas to **forecast data** (not actual data) for the following reasons:

1. **Consistency with comparison methodology**: All scenarios use the same base forecast data for fair comparison
2. **Single perturbation source**: Forecast errors are generated once and applied consistently across all scenarios
3. **Maintain comparable forecast error structure**: Perturbing actual meter readings would give different forecast datasets than perturbing corrected meter readings, making scenario comparisons invalid

**Key Principle**: The forecast errors (difference between actual and forecast) must originate from the same base perturbation across scenarios to isolate the impact of REC sharing on imbalances and costs.

## REC Sharing Formulas

The following formulas calculate how energy is shared within the Renewable Energy Community. **All calculations are performed per timestamp** $t$ (15-minute intervals):

### Aggregate Variables
$$
\text{Total Load}_t = \sum_{i \in \mathcal{R}} L_{i,t}
$$

$$
\text{Total Gen}_t = \sum_{j \in \mathcal{R}} G_{j,t}
$$

$$
\text{Shared Energy}_t = \min(\text{Total Load}_t, \text{Total Gen}_t)
$$

$$
\text{Exported Energy}_t = \max(0, \text{Total Gen}_t - \text{Total Load}_t)
$$

> **Note:** Both Shared Energy and Exported Energy are always non-negative:
> - Shared Energy $= \min(\text{Total Load}_t, \text{Total Gen}_t)$ → Range: $[0, \infty)$
> - Exported Energy $= \max(0, \text{Total Gen}_t - \text{Total Load}_t)$ → Range: $[0, \infty)$

### Corrected Meter Readings

For each consumer $i \in \mathcal{R}$:

$$
L_{i,t}^{corr} = L_{i,t} - \frac{L_{i,t}}{\text{Total Load}_t} \times \text{Shared Energy}_t
$$

For each prosumer $j \in \mathcal{R}$:

$$
G_{j,t}^{corr} = \frac{G_{j,t}}{\text{Total Gen}_t} \times \text{Exported Energy}_t
$$

This can be rewritten in terms of Shared Energy:

$$
G_{j,t}^{corr} = G_{j,t} - \frac{G_{j,t}}{\text{Total Gen}_t} \times \text{Shared Energy}_t
$$

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

# Load configuration from JSON
config_file = Path('../B_Scenarion_Forecasting/B2_multiple_supplier_with_rec.json')
with open(config_file, 'r') as f:
    config = json.load(f)

# Extract REC member IDs from configuration
prosumer_cols = [p['res']['id'] for p in config['prosumers'] if p.get('rec') == 'REC_01']
consumer_cols = [c['load']['id'] for c in config['consumers'] if c.get('rec') == 'REC_01']

print(f"✓ Configuration loaded: {config['recs'][0]['rec_name']}")
print(f"  Prosumers: {len(prosumer_cols)} | Consumers: {len(consumer_cols)} | Total: {len(prosumer_cols) + len(consumer_cols)}")

✓ Configuration loaded: Energy Community 01
  Prosumers: 6 | Consumers: 9 | Total: 15


## Day-Ahead (DA) Forecast Generation

In [2]:
# Load Day-Ahead forecast data
load_forecast_da = pd.read_csv('load_forecast_da.csv', parse_dates=['datetime'], index_col='datetime')
res_forecast_da = pd.read_csv('res_forecast_da.csv', parse_dates=['datetime'], index_col='datetime')

# Extract REC member data
rec_load_da = load_forecast_da[consumer_cols].copy()
rec_res_da = res_forecast_da[prosumer_cols].copy()

# Calculate aggregate variables per timestamp
total_load_da = rec_load_da.sum(axis=1)  # Total Load_t
total_gen_da = rec_res_da.sum(axis=1)    # Total Gen_t
shared_energy_da = np.minimum(total_load_da, total_gen_da)  # Shared Energy_t
exported_energy_da = np.maximum(0, total_gen_da - total_load_da)  # Exported Energy_t

# Apply corrected meter reading formulas
rec_load_corrected_da = rec_load_da.copy()
for col in consumer_cols:
    with np.errstate(divide='ignore', invalid='ignore'):
        # L_i,t^corr = L_i,t - (L_i,t / Total Load_t) × Shared Energy_t
        load_share = np.where(total_load_da > 0, rec_load_da[col] / total_load_da, 0)
    rec_load_corrected_da[col] = rec_load_da[col] - (load_share * shared_energy_da)

rec_res_corrected_da = rec_res_da.copy()
for col in prosumer_cols:
    with np.errstate(divide='ignore', invalid='ignore'):
        # G_j,t^corr = (G_j,t / Total Gen_t) × Exported Energy_t
        gen_share = np.where(total_gen_da > 0, rec_res_da[col] / total_gen_da, 0)
    rec_res_corrected_da[col] = gen_share * exported_energy_da

# Save DA forecasts
rec_load_corrected_da.to_csv('rec_load_forecast_da.csv')
rec_res_corrected_da.to_csv('rec_res_forecast_da.csv')

print(f"✓ Day-Ahead Forecasts Generated")
print(f"  rec_load_forecast_da.csv: {rec_load_corrected_da.shape}")
print(f"  rec_res_forecast_da.csv: {rec_res_corrected_da.shape}")
print(f"  Shared Energy (DA): {shared_energy_da.sum():.2f} kWh")
print(f"  Exported Energy (DA): {exported_energy_da.sum():.2f} kWh")

✓ Day-Ahead Forecasts Generated
  rec_load_forecast_da.csv: (35136, 9)
  rec_res_forecast_da.csv: (35136, 6)
  Shared Energy (DA): 48.31 kWh
  Exported Energy (DA): 113.78 kWh


## Intra-Day (ID) Forecast Generation

In [3]:
# Load Intra-Day forecast data
load_forecast_id = pd.read_csv('load_forecast_id.csv', parse_dates=['datetime'], index_col='datetime')
res_forecast_id = pd.read_csv('res_forecast_id.csv', parse_dates=['datetime'], index_col='datetime')

# Extract REC member data
rec_load_id = load_forecast_id[consumer_cols].copy()
rec_res_id = res_forecast_id[prosumer_cols].copy()

# Calculate aggregate variables per timestamp
total_load_id = rec_load_id.sum(axis=1)  # Total Load_t
total_gen_id = rec_res_id.sum(axis=1)    # Total Gen_t
shared_energy_id = np.minimum(total_load_id, total_gen_id)  # Shared Energy_t
exported_energy_id = np.maximum(0, total_gen_id - total_load_id)  # Exported Energy_t

# Apply corrected meter reading formulas
rec_load_corrected_id = rec_load_id.copy()
for col in consumer_cols:
    with np.errstate(divide='ignore', invalid='ignore'):
        # L_i,t^corr = L_i,t - (L_i,t / Total Load_t) × Shared Energy_t
        load_share = np.where(total_load_id > 0, rec_load_id[col] / total_load_id, 0)
    rec_load_corrected_id[col] = rec_load_id[col] - (load_share * shared_energy_id)

rec_res_corrected_id = rec_res_id.copy()
for col in prosumer_cols:
    with np.errstate(divide='ignore', invalid='ignore'):
        # G_j,t^corr = (G_j,t / Total Gen_t) × Exported Energy_t
        gen_share = np.where(total_gen_id > 0, rec_res_id[col] / total_gen_id, 0)
    rec_res_corrected_id[col] = gen_share * exported_energy_id

# Save ID forecasts
rec_load_corrected_id.to_csv('rec_load_forecast_id.csv')
rec_res_corrected_id.to_csv('rec_res_forecast_id.csv')

print(f"✓ Intra-Day Forecasts Generated")
print(f"  rec_load_forecast_id.csv: {rec_load_corrected_id.shape}")
print(f"  rec_res_forecast_id.csv: {rec_res_corrected_id.shape}")
print(f"  Shared Energy (ID): {shared_energy_id.sum():.2f} kWh")
print(f"  Exported Energy (ID): {exported_energy_id.sum():.2f} kWh")

✓ Intra-Day Forecasts Generated
  rec_load_forecast_id.csv: (35136, 9)
  rec_res_forecast_id.csv: (35136, 6)
  Shared Energy (ID): 45.25 kWh
  Exported Energy (ID): 116.78 kWh


## Validation Summary

In [4]:
print("="*80)
print("VALIDATION SUMMARY")
print("="*80)

# DA Validation
print("\nDay-Ahead (DA):")
print(f"  Min corrected load: {rec_load_corrected_da.min().min():.10f} (should be ≥ 0)")
print(f"  Min corrected gen: {rec_res_corrected_da.min().min():.10f} (should be ≥ 0)")
net_da = rec_load_corrected_da.sum(axis=1) - rec_res_corrected_da.sum(axis=1)
print(f"  Net position mean: {net_da.mean():.4f} kW (+ = import, - = export)")

# ID Validation
print("\nIntra-Day (ID):")
print(f"  Min corrected load: {rec_load_corrected_id.min().min():.10f} (should be ≥ 0)")
print(f"  Min corrected gen: {rec_res_corrected_id.min().min():.10f} (should be ≥ 0)")
net_id = rec_load_corrected_id.sum(axis=1) - rec_res_corrected_id.sum(axis=1)
print(f"  Net position mean: {net_id.mean():.4f} kW (+ = import, - = export)")

print("\n" + "="*80)
print("✓ REC FORECAST GENERATION COMPLETE")
print("="*80)
print("\nGenerated files:")
print("  • rec_load_forecast_da.csv, rec_res_forecast_da.csv")
print("  • rec_load_forecast_id.csv, rec_res_forecast_id.csv")
print("\nThese represent forecasted 'corrected meter readings' after REC internal sharing.")

VALIDATION SUMMARY

Day-Ahead (DA):
  Min corrected load: -0.0000000000 (should be ≥ 0)
  Min corrected gen: 0.0000000000 (should be ≥ 0)
  Net position mean: 0.0001 kW (+ = import, - = export)

Intra-Day (ID):
  Min corrected load: -0.0000000000 (should be ≥ 0)
  Min corrected gen: 0.0000000000 (should be ≥ 0)
  Net position mean: -0.0003 kW (+ = import, - = export)

✓ REC FORECAST GENERATION COMPLETE

Generated files:
  • rec_load_forecast_da.csv, rec_res_forecast_da.csv
  • rec_load_forecast_id.csv, rec_res_forecast_id.csv

These represent forecasted 'corrected meter readings' after REC internal sharing.
